In [2]:
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Import your existing project modules verbatim
from dataloader_anchor import create_dataloaders
from model_anchor import DistanceGatedGeoVDSR

def inspect_trained_checkpoints():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"--- Trust Gate Forensics Scanner | Device: {device} ---")

    # 1. Locate your checkpoints
    CHECKPOINT_DIR = Path(r"D:\Vaibhav\vdsr-atl08\checkpoints_dg_vdsr")
    RUN_NAME = "dg_vdsr_lidar"

    TRAIN_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train",
    ]
    VAL_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous",
    ]

    # Grab all regular epoch checkpoints
    ckpts = sorted(
        CHECKPOINT_DIR.glob(f"{RUN_NAME}_epoch_*.pt"),
        key=lambda x: int(x.stem.split("_epoch_")[-1])
    )

    # Grab the last 3 epochs + the 'best' checkpoint
    target_ckpts = ckpts[-3:] if len(ckpts) >= 3 else ckpts
    best_pt = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
    if best_pt.exists() and best_pt not in target_ckpts:
        target_ckpts.append(best_pt)

    if not target_ckpts:
        print(f"[ERROR] No checkpoints found inside {CHECKPOINT_DIR}")
        return

    print(f"Found {len(target_ckpts)} target checkpoints to scan.\n")

    # 2. Load Validation DataLoader (Batch size 4 keeps memory safe)
    _, val_loader = create_dataloaders(TRAIN_DIRS, VAL_DIRS, batch_size=4)
    model = DistanceGatedGeoVDSR().to(device)

    # 3. Scan each checkpoint
    for ckpt_path in target_ckpts:
        print(f"=========================================================")
        print(f" ANALYZING CHECKPOINT: {ckpt_path.name}")
        print(f"=========================================================")

        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        global_collector = []
        photon_collector = []

        pbar = tqdm(val_loader, desc="Pushing Val Tiles", leave=False)

        with torch.inference_mode():
            for batch in pbar:
                # Unpack standard val structure
                dem_bic_all   = batch['dem_bic'].squeeze(0).to(device)
                lidar_del_all = batch['lidar_delta'].squeeze(0).to(device)
                mask_all      = batch['mask'].squeeze(0).to(device)
                dist_map_all  = batch['dist_map'].squeeze(0).to(device)

                n_patches = dem_bic_all.shape[0]
                mini_batch = 64  # Chunking prevents Quadro P4000 VRAM overflow

                for start in range(0, n_patches, mini_batch):
                    end = min(start + mini_batch, n_patches)

                    b_dem = dem_bic_all[start:end]
                    b_del = lidar_del_all[start:end]
                    b_msk = mask_all[start:end]
                    b_dst = dist_map_all[start:end]

                    _, alpha, _, _ = model(b_dem, b_del, b_msk, b_dst)

                    # Flatten [B, 1, 256, 256] -> 1D CPU array
                    a_flat = alpha.detach().view(-1).cpu()
                    m_flat = b_msk.detach().view(-1).cpu()

                    global_collector.append(a_flat)

                    # Isolate strictly the pixels where a photon exists
                    photon_idx = (m_flat > 0)
                    if photon_idx.any():
                        photon_collector.append(a_flat[photon_idx])

        # Concatenate into master RAM arrays
        G = torch.cat(global_collector, dim=0).numpy().astype(np.float64)
        P = torch.cat(photon_collector, dim=0).numpy().astype(np.float64) if photon_collector else np.array([])

        def print_percentiles(arr, title):
            if len(arr) == 0:
                print(f" [{title}] No pixels captured.")
                return

            q = np.percentile(arr, [25, 50, 75, 90, 99])
            print(f" --- {title} (Sampled Pixels: {len(arr):,}) ---")
            print(f"  mean = {arr.mean():.4f}")
            print(f"  min  = {arr.min():.4f}")
            print(f"  25%  = {q[0]:.4f}")
            print(f"  50%  = {q[1]:.4f}")
            print(f"  75%  = {q[2]:.4f}")
            print(f"  90%  = {q[3]:.4f}")
            print(f"  99%  = {q[4]:.4f}")
            print(f"  max  = {arr.max():.4f}\n")

        print_percentiles(G, "GLOBAL MAP (All Terrain)")
        print_percentiles(P, "PHOTON TRACKS ONLY (Where Mask == 1)")

if __name__ == '__main__':
    inspect_trained_checkpoints()

--- Trust Gate Forensics Scanner | Device: cuda ---
Found 4 target checkpoints to scan.

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_498.pt


C:\Users\RRSCW\AppData\Local\Temp\ipykernel_24168\904125909.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  mean = 0.0206
  min  = 0.0000
  25%  = 0.0000
  50%  = 0.0000
  75%  = 0.0002
  90%  = 0.0223
  99%  = 0.5208
  max  = 0.9983

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  mean = 0.5884
  min  = 0.0978
  25%  = 0.4289
  50%  = 0.5823
  75%  = 0.7877
  90%  = 0.9010
  99%  = 0.9802
  max  = 0.9983

 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_499.pt


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  mean = 0.0201
  min  = 0.0000
  25%  = 0.0000
  50%  = 0.0000
  75%  = 0.0002
  90%  = 0.0216
  99%  = 0.5089
  max  = 0.9982

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  mean = 0.5811
  min  = 0.0968
  25%  = 0.4211
  50%  = 0.5720
  75%  = 0.7786
  90%  = 0.8954
  99%  = 0.9784
  max  = 0.9982

 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_500.pt


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  mean = 0.0199
  min  = 0.0000
  25%  = 0.0000
  50%  = 0.0000
  75%  = 0.0002
  90%  = 0.0213
  99%  = 0.5044
  max  = 0.9981

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  mean = 0.5781
  min  = 0.0964
  25%  = 0.4184
  50%  = 0.5679
  75%  = 0.7750
  90%  = 0.8933
  99%  = 0.9778
  max  = 0.9981

 ANALYZING CHECKPOINT: dg_vdsr_lidar_best.pt


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  mean = 0.0265
  min  = 0.0000
  25%  = 0.0000
  50%  = 0.0000
  75%  = 0.0011
  90%  = 0.0411
  99%  = 0.6177
  max  = 0.9999

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  mean = 0.5884
  min  = 0.0366
  25%  = 0.3715
  50%  = 0.6010
  75%  = 0.8197
  90%  = 0.9343
  99%  = 0.9948
  max  = 0.9999



In [ ]:
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Import your existing project modules verbatim
from dataloader_anchor import create_dataloaders
from model_anchor import DistanceGatedGeoVDSR

def inspect_trained_checkpoints():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"--- Trust Gate Forensics Scanner | Device: {device} ---")

    # 1. Locate your checkpoints
    CHECKPOINT_DIR = Path(r"D:\Vaibhav\vdsr-atl08\checkpoints_dg_vdsr")
    RUN_NAME = "dg_vdsr_lidar"

    TRAIN_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train",
    ]
    VAL_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous",
    ]

    # Grab all regular epoch checkpoints
    ckpts = sorted(
        CHECKPOINT_DIR.glob(f"{RUN_NAME}_epoch_*.pt"),
        key=lambda x: int(x.stem.split("_epoch_")[-1])
    )

    # Grab the last 3 epochs + the 'best' checkpoint
    target_ckpts = ckpts[-3:] if len(ckpts) >= 3 else ckpts
    best_pt = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
    if best_pt.exists() and best_pt not in target_ckpts:
        target_ckpts.append(best_pt)

    if not target_ckpts:
        print(f"[ERROR] No checkpoints found inside {CHECKPOINT_DIR}")
        return

    print(f"Found {len(target_ckpts)} target checkpoints to scan.\n")

    # 2. Load Validation DataLoader (Batch size 4 keeps memory safe)
    _, val_loader = create_dataloaders(TRAIN_DIRS, VAL_DIRS, batch_size=4) #[cite: 3]
    model = DistanceGatedGeoVDSR().to(device) #[cite: 4]

    # 3. Scan each checkpoint
    for ckpt_path in target_ckpts:
        print(f"=========================================================")
        print(f" ANALYZING CHECKPOINT: {ckpt_path.name}")
        print(f"=========================================================")

        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        global_collector = []
        photon_collector = []

        pbar = tqdm(val_loader, desc="Pushing Val Tiles", leave=False)

        with torch.inference_mode():
            for batch in pbar:
                # Unpack standard val structure
                dem_bic_all   = batch['dem_bic'].squeeze(0).to(device) #[cite: 3]
                lidar_del_all = batch['lidar_delta'].squeeze(0).to(device) #[cite: 3]
                mask_all      = batch['mask'].squeeze(0).to(device) #[cite: 3]
                dist_map_all  = batch['dist_map'].squeeze(0).to(device) #[cite: 3]

                n_patches = dem_bic_all.shape[0]
                mini_batch = 64  # Chunking prevents Quadro P4000 VRAM overflow

                for start in range(0, n_patches, mini_batch):
                    end = min(start + mini_batch, n_patches)

                    b_dem = dem_bic_all[start:end]
                    b_del = lidar_del_all[start:end]
                    b_msk = mask_all[start:end]
                    b_dst = dist_map_all[start:end]

                    _, alpha, _, _ = model(b_dem, b_del, b_msk, b_dst) #[cite: 4]

                    # Flatten [B, 1, 256, 256] -> 1D CPU array
                    a_flat = alpha.detach().view(-1).cpu()
                    m_flat = b_msk.detach().view(-1).cpu()

                    global_collector.append(a_flat)

                    # Isolate strictly the pixels where a photon exists[cite: 3]
                    photon_idx = (m_flat > 0)
                    if photon_idx.any():
                        photon_collector.append(a_flat[photon_idx])

        # Concatenate into master RAM arrays
        G = torch.cat(global_collector, dim=0).numpy().astype(np.float64)
        P = torch.cat(photon_collector, dim=0).numpy().astype(np.float64) if photon_collector else np.array([])

        def print_percentiles(arr, title):
            if len(arr) == 0:
                print(f" [{title}] No pixels captured.")
                return

            # Generate dynamic list: [5, 10, 15, ..., 90, 95, 99]
            p_list = list(range(5, 100, 5)) + [99]
            q_values = np.percentile(arr, p_list)

            print(f" --- {title} (Sampled Pixels: {len(arr):,}) ---")
            print(f"  mean = {arr.mean():.4f}")
            print(f"  min  = {arr.min():.4f}")

            # Loop cleanly through our dynamic intervals
            for p, val in zip(p_list, q_values):
                print(f"  {p:2d}%  = {val:.4f}")

            print(f"  max  = {arr.max():.4f}\n")

        print_percentiles(G, "GLOBAL MAP (All Terrain)")
        print_percentiles(P, "PHOTON TRACKS ONLY (Where Mask == 1)")

if __name__ == '__main__':
    inspect_trained_checkpoints()

--- Trust Gate Forensics Scanner | Device: cuda ---
Found 4 target checkpoints to scan.

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_498.pt


C:\Users\RRSCW\AppData\Local\Temp\ipykernel_20996\1719761731.py:55: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  mean = 0.0206
  min  = 0.0000
   5%  = 0.0000
  10%  = 0.0000
  15%  = 0.0000
  20%  = 0.0000
  25%  = 0.0000
  30%  = 0.0000
  35%  = 0.0000
  40%  = 0.0000
  45%  = 0.0000
  50%  = 0.0000
  55%  = 0.0000
  60%  = 0.0000
  65%  = 0.0000
  70%  = 0.0000
  75%  = 0.0002
  80%  = 0.0008
  85%  = 0.0040
  90%  = 0.0223
  95%  = 0.1119
  99%  = 0.5208
  max  = 0.9983

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  mean = 0.5884
  min  = 0.0978
   5%  = 0.2015
  10%  = 0.2503
  15%  = 0.3083
  20%  = 0.3726
  25%  = 0.4289
  30%  = 0.4691
  35%  = 0.5023
  40%  = 0.5316
  45%  = 0.5580
  50%  = 0.5823
  55%  = 0.6093
  60%  = 0.6464
  65%  = 0.6923
  70%  = 0.7413
  75%  = 0.7877
  80%  = 0.8222
  85%  = 0.8523
  90%  = 0.9010
  95%  = 0.9427
  99%  = 0.9802
  max  = 0.9983

 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_499.pt


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  mean = 0.0201
  min  = 0.0000
   5%  = 0.0000
  10%  = 0.0000
  15%  = 0.0000
  20%  = 0.0000
  25%  = 0.0000
  30%  = 0.0000
  35%  = 0.0000
  40%  = 0.0000
  45%  = 0.0000
  50%  = 0.0000
  55%  = 0.0000
  60%  = 0.0000
  65%  = 0.0000
  70%  = 0.0000
  75%  = 0.0002
  80%  = 0.0008
  85%  = 0.0038
  90%  = 0.0216
  95%  = 0.1088
  99%  = 0.5089
  max  = 0.9982

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  mean = 0.5811
  min  = 0.0968
   5%  = 0.1992
  10%  = 0.2470
  15%  = 0.3036
  20%  = 0.3658
  25%  = 0.4211
  30%  = 0.4605
  35%  = 0.4933
  40%  = 0.5224
  45%  = 0.5477
  50%  = 0.5720
  55%  = 0.6000
  60%  = 0.6360
  65%  = 0.6828
  70%  = 0.7325
  75%  = 0.7786
  80%  = 0.8135
  85%  = 0.8447
  90%  = 0.8954
  95%  = 0.9387
  99%  = 0.9784
  max  = 0.9982

 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_500.pt


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

In [ ]:
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

# Import your existing project modules verbatim
from dataloader_anchor import create_dataloaders
from model_anchor import DistanceGatedGeoVDSR

def inspect_trained_checkpoints():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"--- Trust Gate Forensics Scanner | Device: {device} ---")

    CHECKPOINT_DIR = Path(r"D:\Vaibhav\vdsr-atl08\checkpoints_dg_vdsr")
    RUN_NAME = "dg_vdsr_lidar"

    TRAIN_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train",
    ]
    VAL_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous",
    ]

    ckpts = sorted(
        CHECKPOINT_DIR.glob(f"{RUN_NAME}_epoch_*.pt"),
        key=lambda x: int(x.stem.split("_epoch_")[-1])
    )

    target_ckpts = ckpts[-3:] if len(ckpts) >= 3 else ckpts
    best_pt = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
    if best_pt.exists() and best_pt not in target_ckpts:
        target_ckpts.append(best_pt)

    if not target_ckpts:
        print(f"[ERROR] No checkpoints found inside {CHECKPOINT_DIR}")
        return

    print(f"Found {len(target_ckpts)} target checkpoints to scan.\n")

    _, val_loader = create_dataloaders(TRAIN_DIRS, VAL_DIRS, batch_size=4) #[cite: 3]
    model = DistanceGatedGeoVDSR().to(device) #

    for ckpt_path in target_ckpts:
        print(f"=========================================================")
        print(f" ANALYZING CHECKPOINT: {ckpt_path.name}")
        print(f"=========================================================")

        checkpoint = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        # Storage buckets for our 4 tracked metrics
        global_alpha, global_topo, global_anch, global_fin = [], [], [], []
        photon_alpha, photon_topo, photon_anch, photon_fin = [], [], [], []

        pbar = tqdm(val_loader, desc="Pushing Val Tiles", leave=False)

        with torch.inference_mode():
            for batch in pbar:
                dem_bic_all   = batch['dem_bic'].squeeze(0).to(device) #[cite: 3]
                lidar_del_all = batch['lidar_delta'].squeeze(0).to(device) #[cite: 3]
                mask_all      = batch['mask'].squeeze(0).to(device) #[cite: 3]
                dist_map_all  = batch['dist_map'].squeeze(0).to(device) #[cite: 3]

                n_patches = dem_bic_all.shape[0]
                mini_batch = 64  

                for start in range(0, n_patches, mini_batch):
                    end = min(start + mini_batch, n_patches)

                    b_dem = dem_bic_all[start:end]
                    b_del = lidar_del_all[start:end]
                    b_msk = mask_all[start:end]
                    b_dst = dist_map_all[start:end]

                    # 1. Unpack all 3 returned branches
                    _, alpha, r_topo, r_anchor = model(b_dem, b_del, b_msk, b_dst) #[cite: 4]

                    # 2. Formulate r_final explicitly
                    r_final = (1.0 - alpha) * r_topo + (alpha * r_anchor)

                    # Flatten to 1D CPU arrays (float32 saves RAM during collection)
                   # Flatten to 1D CPU arrays (.reshape() handles strided broadcast memory)
                    a_flat = alpha.detach().reshape(-1).cpu()
                    t_flat = r_topo.detach().reshape(-1).cpu()
                    n_flat = r_anchor.detach().reshape(-1).cpu()
                    f_flat = r_final.detach().reshape(-1).cpu()
                    m_flat = b_msk.detach().reshape(-1).cpu()

                    # Push to Global Collector
                    global_alpha.append(a_flat)
                    global_topo.append(t_flat)
                    global_anch.append(n_flat)
                    global_fin.append(f_flat)

                    # Push to Photon-Only Collector[cite: 3]
                    photon_idx = (m_flat > 0)
                    if photon_idx.any():
                        photon_alpha.append(a_flat[photon_idx])
                        photon_topo.append(t_flat[photon_idx])
                        photon_anch.append(n_flat[photon_idx])
                        photon_fin.append(f_flat[photon_idx])

        # Convert collectors to master float64 NumPy arrays
        G_a = torch.cat(global_alpha, dim=0).numpy().astype(np.float64)
        G_t = torch.cat(global_topo,  dim=0).numpy().astype(np.float64)
        G_n = torch.cat(global_anch,  dim=0).numpy().astype(np.float64)
        G_f = torch.cat(global_fin,   dim=0).numpy().astype(np.float64)

        if photon_alpha:
            P_a = torch.cat(photon_alpha, dim=0).numpy().astype(np.float64)
            P_t = torch.cat(photon_topo,  dim=0).numpy().astype(np.float64)
            P_n = torch.cat(photon_anch,  dim=0).numpy().astype(np.float64)
            P_f = torch.cat(photon_fin,   dim=0).numpy().astype(np.float64)
        else:
            P_a = P_t = P_n = P_f = np.array([])

        def print_forensics(a_arr, t_arr, n_arr, f_arr, title):
            if len(a_arr) == 0:
                print(f" [{title}] No pixels captured.\n")
                return

            p_list = list(range(5, 100, 5)) + [99]
            q_values = np.percentile(a_arr, p_list)

            print(f" --- {title} (Sampled Pixels: {len(a_arr):,}) ---")
            print(f"  [Residual Magnitudes (|Δ| from Bicubic Baseline)]")
            print(f"   |r_topo|   abs_mean = {np.abs(t_arr).mean():.4f} m")
            print(f"   |r_anchor| abs_mean = {np.abs(n_arr).mean():.4f} m")
            print(f"   |r_final|  abs_mean = {np.abs(f_arr).mean():.4f} m\n")

            print(f"  [Trust Gate (Alpha) Quantiles]")
            print(f"   mean = {a_arr.mean():.4f}")
            print(f"   min  = {a_arr.min():.4f}")

            for p, val in zip(p_list, q_values):
                print(f"   {p:2d}%  = {val:.4f}")

            print(f"   max  = {a_arr.max():.4f}\n")

        print_forensics(G_a, G_t, G_n, G_f, "GLOBAL MAP (All Terrain)")
        print_forensics(P_a, P_t, P_n, P_f, "PHOTON TRACKS ONLY (Where Mask == 1)")

if __name__ == '__main__':
    inspect_trained_checkpoints()

--- Trust Gate Forensics Scanner | Device: cuda ---
Found 4 target checkpoints to scan.

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_498.pt


C:\Users\RRSCW\AppData\Local\Temp\ipykernel_24524\423230458.py:50: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(ckpt_path, map_location=device)


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 449,445,888) ---
  [Residual Magnitudes (|Δ| from Bicubic Baseline)]
   |r_topo|   abs_mean = 0.4418 m
   |r_anchor| abs_mean = 2.9862 m
   |r_final|  abs_mean = 0.4376 m

  [Trust Gate (Alpha) Quantiles]
   mean = 0.0206
   min  = 0.0000
    5%  = 0.0000
   10%  = 0.0000
   15%  = 0.0000
   20%  = 0.0000
   25%  = 0.0000
   30%  = 0.0000
   35%  = 0.0000
   40%  = 0.0000
   45%  = 0.0000
   50%  = 0.0000
   55%  = 0.0000
   60%  = 0.0000
   65%  = 0.0000
   70%  = 0.0000
   75%  = 0.0002
   80%  = 0.0008
   85%  = 0.0040
   90%  = 0.0223
   95%  = 0.1119
   99%  = 0.5208
   max  = 0.9983

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  [Residual Magnitudes (|Δ| from Bicubic Baseline)]
   |r_topo|   abs_mean = 0.1958 m
   |r_anchor| abs_mean = 0.6201 m
   |r_final|  abs_mean = 0.4382 m

  [Trust Gate (Alpha) Quantiles]
   mean = 0.5884
   min  = 0.0978
    5%  = 0.2015
   10%  = 0.2503
   15%  = 0.3083
   20%  = 0

Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

In [1]:
import torch
import numpy as np
from pathlib import Path
from tqdm.auto import tqdm

from dataloader_anchor import create_dataloaders
from model_anchor import DistanceGatedGeoVDSR

def inspect_trained_checkpoints():
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"--- Trust Gate Forensics Scanner | Device: {device} ---")

    CHECKPOINT_DIR = Path(r"D:\Vaibhav\vdsr-atl08\checkpoints_dg_vdsr")
    RUN_NAME = "dg_vdsr_lidar"

    TRAIN_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train",
    ]
    VAL_DIRS = [
        r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
        r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous",
    ]

    ckpts = sorted(
        CHECKPOINT_DIR.glob(f"{RUN_NAME}_epoch_*.pt"),
        key=lambda x: int(x.stem.split("_epoch_")[-1])
    )

    target_ckpts = ckpts[-3:] if len(ckpts) >= 3 else ckpts
    best_pt = CHECKPOINT_DIR / f"{RUN_NAME}_best.pt"
    if best_pt.exists() and best_pt not in target_ckpts:
        target_ckpts.append(best_pt)

    if not target_ckpts:
        print(f"[ERROR] No checkpoints found inside {CHECKPOINT_DIR}")
        return

    print(f"Found {len(target_ckpts)} target checkpoints to scan.\n")

    _, val_loader = create_dataloaders(TRAIN_DIRS, VAL_DIRS, batch_size=4)
    model = DistanceGatedGeoVDSR().to(device)

    for ckpt_path in target_ckpts:
        print(f"=========================================================")
        print(f" ANALYZING CHECKPOINT: {ckpt_path.name}")
        print(f"=========================================================")

        # Explicit weights_only=False silences the PyTorch 2.4+ security warning
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
        model.eval()

        G_alp, G_top, G_anc, G_fin = [], [], [], []
        G_ebase, G_etopo, G_eanch, G_efinal = [], [], [], []

        P_alp, P_top, P_anc, P_fin = [], [], [], []
        P_ebase, P_etopo, P_eanch, P_efinal = [], [], [], []
        P_eoffset = []  # <-- FIXED: Initialized accumulator list
        P_etrue_anc = []

        pbar = tqdm(val_loader, desc="Pushing Val Tiles", leave=False)

        with torch.inference_mode():
            for batch in pbar:
                dem_bic_all   = batch['dem_bic'].squeeze(0).to(device)
                gt_dem_all    = batch['gt_dem'].squeeze(0).to(device)
                lidar_del_all = batch['lidar_delta'].squeeze(0).to(device)
                mask_all      = batch['mask'].squeeze(0).to(device)
                dist_map_all  = batch['dist_map'].squeeze(0).to(device)

                n_patches = dem_bic_all.shape[0]
                mini_batch = 64  

                for start in range(0, n_patches, mini_batch):
                    end = min(start + mini_batch, n_patches)

                    b_dem = dem_bic_all[start:end]
                    b_gt  = gt_dem_all[start:end]
                    b_del = lidar_del_all[start:end]
                    b_msk = mask_all[start:end]
                    b_dst = dist_map_all[start:end]

                    _, alpha, r_topo, r_anchor = model(b_dem, b_del, b_msk, b_dst)

                    r_final = (1.0 - alpha) * r_topo + (alpha * r_anchor)
                    e_offset = (b_dem + b_del - b_gt).abs()

                    e_base  = (b_dem - b_gt).abs()
                    e_topo  = (b_dem + r_topo - b_gt).abs()
                    e_anch  = (b_dem + r_anchor - b_gt).abs()
                    e_fin   = (b_dem + r_final - b_gt).abs()
                    e_true_anchor = (r_anchor - b_del).abs()

                    a_flat  = alpha.detach().reshape(-1).cpu()
                    t_flat  = r_topo.detach().reshape(-1).cpu()
                    n_flat  = r_anchor.detach().reshape(-1).cpu()
                    f_flat  = r_final.detach().reshape(-1).cpu()
                    m_flat  = b_msk.detach().reshape(-1).cpu()

                    eb_flat = e_base.detach().reshape(-1).cpu()
                    et_flat = e_topo.detach().reshape(-1).cpu()
                    en_flat = e_anch.detach().reshape(-1).cpu()
                    ef_flat = e_fin.detach().reshape(-1).cpu()
                    

                    # <-- THE RAM SAVER: Subsample empty space by 25x
                    step = 25
                    G_alp.append(a_flat[::step]); G_top.append(t_flat[::step]); G_anc.append(n_flat[::step]); G_fin.append(f_flat[::step])
                    G_ebase.append(eb_flat[::step]); G_etopo.append(et_flat[::step]); G_eanch.append(en_flat[::step]); G_efinal.append(ef_flat[::step])

                    # <-- THE PRECISION PRESERVER: Keep 100% of physical photons
                    photon_idx = (m_flat > 0)
                    if photon_idx.any():
                        P_alp.append(a_flat[photon_idx]); P_top.append(t_flat[photon_idx])
                        P_anc.append(n_flat[photon_idx]); P_fin.append(f_flat[photon_idx])
                        
                        P_ebase.append(eb_flat[photon_idx]); P_etopo.append(et_flat[photon_idx])
                        P_eanch.append(en_flat[photon_idx]); P_efinal.append(ef_flat[photon_idx])
                        P_eoffset.append(e_offset.detach().reshape(-1).cpu()[photon_idx])
                        P_etrue_anc.append(e_true_anchor.detach().reshape(-1).cpu()[photon_idx])

        # Cast to float32 (plenty for 4 decimal places, saves 50% RAM over float64)
        ga = torch.cat(G_alp, dim=0).numpy().astype(np.float32)
        gt = torch.cat(G_top, dim=0).numpy().astype(np.float32)
        gn = torch.cat(G_anc, dim=0).numpy().astype(np.float32)
        gf = torch.cat(G_fin, dim=0).numpy().astype(np.float32)
        geb = torch.cat(G_ebase, dim=0).numpy().astype(np.float32)
        get = torch.cat(G_etopo, dim=0).numpy().astype(np.float32)
        gen = torch.cat(G_eanch, dim=0).numpy().astype(np.float32)
        gef = torch.cat(G_efinal, dim=0).numpy().astype(np.float32)

        if P_alp:
            pa = torch.cat(P_alp, dim=0).numpy().astype(np.float32)
            pt = torch.cat(P_top, dim=0).numpy().astype(np.float32)
            pn = torch.cat(P_anc, dim=0).numpy().astype(np.float32)
            pf = torch.cat(P_fin, dim=0).numpy().astype(np.float32)
            peb = torch.cat(P_ebase, dim=0).numpy().astype(np.float32)
            pet = torch.cat(P_etopo, dim=0).numpy().astype(np.float32)
            pen = torch.cat(P_eanch, dim=0).numpy().astype(np.float32)
            pef = torch.cat(P_efinal, dim=0).numpy().astype(np.float32)
            peoffset = torch.cat(P_eoffset, dim=0).numpy().astype(np.float32)  # <-- FIXED: Concatenated offset array
            petrue_anc = torch.cat(P_etrue_anc, dim=0).numpy().astype(np.float32)  # <-- STEP 1 ADDITION
        else:
            pa = pt = pn = pf = peb = pet = pen = pef = peoffset = petrue_anc = np.array([])

        def print_forensics(a_arr, t_arr, n_arr, f_arr, eb, et, en, ef, title):
            if len(a_arr) == 0:
                print(f" [{title}] No pixels captured.\n")
                return

            p_list = list(range(5, 100, 5)) + [99]
            q_values = np.percentile(a_arr, p_list)

            print(f" --- {title} (Sampled Pixels: {len(a_arr):,}) ---")
            
            print(f"  [Mean Absolute Error vs. Ground Truth DEM]")
            print(f"   base_mae   = {eb.mean():.4f} m")
            print(f"   topo_mae   = {et.mean():.4f} m")
            print(f"   anchor_mae = {en.mean():.4f} m")
            print(f"   final_mae  = {ef.mean():.4f} m\n")

            print(f"  [Residual Magnitudes (|Δ| from Bicubic Baseline)]")
            print(f"   |r_topo|   abs_mean = {np.abs(t_arr).mean():.4f} m")
            print(f"   |r_anchor| abs_mean = {np.abs(n_arr).mean():.4f} m")
            print(f"   |r_final|  abs_mean = {np.abs(f_arr).mean():.4f} m\n")

            print(f"  [Trust Gate (Alpha) Quantiles]")
            print(f"   mean = {a_arr.mean():.4f}")
            print(f"   min  = {a_arr.min():.4f}")

            for p, val in zip(p_list, q_values):
                print(f"   {p:2d}%  = {val:.4f}")

            print(f"   max  = {a_arr.max():.4f}\n")

        print_forensics(ga, gt, gn, gf, geb, get, gen, gef, "GLOBAL MAP (All Terrain)")
        print_forensics(pa, pt, pn, pf, peb, pet, pen, pef, "PHOTON TRACKS ONLY (Where Mask == 1)")
        
        if len(peoffset) > 0:
            print(f"   PHYSICAL DATUM OFFSET (Raw LiDAR vs GT DEM) = {peoffset.mean():.4f} m")
            # --- STEP 1 ADDITION ---
            print(f"   TRUE ANCHOR STREAM ERROR (vs Raw LiDAR Target) = {petrue_anc.mean():.4f} m\n")
        else:
            print(f"   PHYSICAL DATUM OFFSET (Raw LiDAR vs GT DEM) = N/A\n")

if __name__ == '__main__':
    inspect_trained_checkpoints()

--- Trust Gate Forensics Scanner | Device: cuda ---
Found 4 target checkpoints to scan.

[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.
 ANALYZING CHECKPOINT: dg_vdsr_lidar_epoch_498.pt


Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 17,977,929) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 4.6855 m
   topo_mae   = 4.6329 m
   anchor_mae = 5.5524 m
   final_mae  = 4.6329 m

  [Residual Magnitudes (|Δ| from Bicubic Baseline)]
   |r_topo|   abs_mean = 0.4418 m
   |r_anchor| abs_mean = 2.9862 m
   |r_final|  abs_mean = 0.4376 m

  [Trust Gate (Alpha) Quantiles]
   mean = 0.0206
   min  = 0.0000
    5%  = 0.0000
   10%  = 0.0000
   15%  = 0.0000
   20%  = 0.0000
   25%  = 0.0000
   30%  = 0.0000
   35%  = 0.0000
   40%  = 0.0000
   45%  = 0.0000
   50%  = 0.0000
   55%  = 0.0000
   60%  = 0.0000
   65%  = 0.0000
   70%  = 0.0000
   75%  = 0.0002
   80%  = 0.0008
   85%  = 0.0040
   90%  = 0.0223
   95%  = 0.1119
   99%  = 0.5208
   max  = 0.9970

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 1.4687 m
   topo_mae   = 1.4667 m
   anchor_mae = 1.5268 m
   final_mae  = 1.4

Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 17,977,929) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 4.6855 m
   topo_mae   = 4.6492 m
   anchor_mae = 5.5754 m
   final_mae  = 4.6490 m

  [Residual Magnitudes (|Δ| from Bicubic Baseline)]
   |r_topo|   abs_mean = 0.4437 m
   |r_anchor| abs_mean = 3.0297 m
   |r_final|  abs_mean = 0.4395 m

  [Trust Gate (Alpha) Quantiles]
   mean = 0.0201
   min  = 0.0000
    5%  = 0.0000
   10%  = 0.0000
   15%  = 0.0000
   20%  = 0.0000
   25%  = 0.0000
   30%  = 0.0000
   35%  = 0.0000
   40%  = 0.0000
   45%  = 0.0000
   50%  = 0.0000
   55%  = 0.0000
   60%  = 0.0000
   65%  = 0.0000
   70%  = 0.0000
   75%  = 0.0002
   80%  = 0.0008
   85%  = 0.0038
   90%  = 0.0216
   95%  = 0.1087
   99%  = 0.5089
   max  = 0.9969

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 1.4687 m
   topo_mae   = 1.4696 m
   anchor_mae = 1.5255 m
   final_mae  = 1.4

Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 17,977,929) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 4.6855 m
   topo_mae   = 4.6661 m
   anchor_mae = 5.5459 m
   final_mae  = 4.6658 m

  [Residual Magnitudes (|Δ| from Bicubic Baseline)]
   |r_topo|   abs_mean = 0.4358 m
   |r_anchor| abs_mean = 2.9746 m
   |r_final|  abs_mean = 0.4318 m

  [Trust Gate (Alpha) Quantiles]
   mean = 0.0199
   min  = 0.0000
    5%  = 0.0000
   10%  = 0.0000
   15%  = 0.0000
   20%  = 0.0000
   25%  = 0.0000
   30%  = 0.0000
   35%  = 0.0000
   40%  = 0.0000
   45%  = 0.0000
   50%  = 0.0000
   55%  = 0.0000
   60%  = 0.0000
   65%  = 0.0000
   70%  = 0.0000
   75%  = 0.0002
   80%  = 0.0007
   85%  = 0.0038
   90%  = 0.0214
   95%  = 0.1074
   99%  = 0.5044
   max  = 0.9967

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 1.4687 m
   topo_mae   = 1.4741 m
   anchor_mae = 1.5259 m
   final_mae  = 1.4

Pushing Val Tiles:   0%|          | 0/6 [00:00<?, ?it/s]

 --- GLOBAL MAP (All Terrain) (Sampled Pixels: 17,977,929) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 4.6855 m
   topo_mae   = 4.4991 m
   anchor_mae = 6.0633 m
   final_mae  = 4.5009 m

  [Residual Magnitudes (|Δ| from Bicubic Baseline)]
   |r_topo|   abs_mean = 0.5184 m
   |r_anchor| abs_mean = 3.8753 m
   |r_final|  abs_mean = 0.5097 m

  [Trust Gate (Alpha) Quantiles]
   mean = 0.0265
   min  = 0.0000
    5%  = 0.0000
   10%  = 0.0000
   15%  = 0.0000
   20%  = 0.0000
   25%  = 0.0000
   30%  = 0.0000
   35%  = 0.0000
   40%  = 0.0000
   45%  = 0.0000
   50%  = 0.0000
   55%  = 0.0000
   60%  = 0.0000
   65%  = 0.0001
   70%  = 0.0004
   75%  = 0.0011
   80%  = 0.0034
   85%  = 0.0117
   90%  = 0.0411
   95%  = 0.1530
   99%  = 0.6179
   max  = 0.9999

 --- PHOTON TRACKS ONLY (Where Mask == 1) (Sampled Pixels: 552,669) ---
  [Mean Absolute Error vs. Ground Truth DEM]
   base_mae   = 1.4687 m
   topo_mae   = 1.4228 m
   anchor_mae = 1.5048 m
   final_mae  = 1.4

In [1]:
import torch
import numpy as np
from tqdm.auto import tqdm
from dataloader_anchor import create_dataloaders

TRAIN_DIRS = [
    r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\train",
    r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\train",
]
VAL_DIRS = [
    r"D:\Vaibhav\vdsr-atl08\Dataset\Kl\tensors\validation_contiguous",
    r"D:\Vaibhav\vdsr-atl08\Dataset\SG\tensors\validation_contiguous",
]

print("--- GEODETIC DATUM FORENSICS (CPU Direct) ---")
_, val_loader = create_dataloaders(TRAIN_DIRS, VAL_DIRS, batch_size=1)

raw_diffs = []

for batch in tqdm(val_loader, desc="Scanning Dataset Geometry"):
    # Unpack CPU tensors directly
    dem  = batch['dem_bic'].squeeze(0)
    gt   = batch['gt_dem'].squeeze(0)
    del_ = batch['lidar_delta'].squeeze(0)
    mask = batch['mask'].squeeze(0)

    lidar_true = dem + del_
    
    # Calculate SIGNED difference strictly at photon hits
    diff = (lidar_true - gt)[mask > 0]
    
    if len(diff) > 0:
        raw_diffs.append(diff.numpy())

D = np.concatenate(raw_diffs, axis=0)

print(f"\n Captured {len(D):,} physical LiDAR photons across validation map:")
print(f" -------------------------------------------------------")
print(f"  Mean Signed Offset:   {D.mean():+.4f} meters")
print(f"  Median Signed Offset: {np.median(D):+.4f} meters")
print(f"  Standard Deviation:   {D.std():.4f} meters")
print(f"  Min / Max:            {D.min():+.2f}m  to  {D.max():+.2f}m")

--- GEODETIC DATUM FORENSICS (CPU Direct) ---
[TRAIN DATASET] Loaded 1649 training files.
   -> Generating 2048x2048 Spatial Noise Cache in CPU RAM...
[VAL DATASET] Loaded 6 validation files.


Scanning Dataset Geometry:   0%|          | 0/6 [00:00<?, ?it/s]


 Captured 552,669 physical LiDAR photons across validation map:
 -------------------------------------------------------
  Mean Signed Offset:   -0.4069 meters
  Median Signed Offset: -0.4329 meters
  Standard Deviation:   2.4822 meters
  Min / Max:            -8.00m  to  +7.99m


In [2]:
import os
import glob
import numpy as np
from tqdm.notebook import tqdm

# Define your training data directory/directories


In [3]:
import os
import numpy as np
from tqdm.notebook import tqdm

# UPDATE THIS PATH to your parent directory
PARENT_DIR = r"D:\Vaibhav\vdsr-atl08\Dataset"

def calculate_dataset_stats(parent_dir):
    filepaths = []
    for root, _, files in os.walk(parent_dir):
        for f in files:
            if f.endswith('.npy'):
                filepaths.append(os.path.join(root, f))
                
    print(f"Found {len(filepaths)} files in '{parent_dir}' and its subdirectories. Calculating statistics...")
    
    global_min = np.inf
    global_max = -np.inf
    
    dem_min, dem_max = np.inf, -np.inf
    gt_min, gt_max = np.inf, -np.inf
    lidar_min, lidar_max = np.inf, -np.inf
    
    count = 0
    sum_val = 0.0
    sum_sq_val = 0.0
    
    for path in tqdm(filepaths, desc="Scanning Dataset"):
        data = np.load(path).astype(np.float32)
        
        if data.ndim == 2:
            data = data[np.newaxis, :, :]
            
        dem_bic = data[0]
        lidar   = data[1]
        mask    = data[2]
        gt_dem  = data[3]
        
        dem_min = min(dem_min, float(np.min(dem_bic)))
        dem_max = max(dem_max, float(np.max(dem_bic)))
        
        gt_min = min(gt_min, float(np.min(gt_dem)))
        gt_max = max(gt_max, float(np.max(gt_dem)))
        
        valid_lidar = lidar[mask > 0]
        if len(valid_lidar) > 0:
            lidar_min = min(lidar_min, float(np.min(valid_lidar)))
            lidar_max = max(lidar_max, float(np.max(valid_lidar)))
            
        file_min = min(float(np.min(dem_bic)), float(np.min(gt_dem)))
        file_max = max(float(np.max(dem_bic)), float(np.max(gt_dem)))
        
        global_min = min(global_min, file_min)
        global_max = max(global_max, file_max)
        
        sum_val += np.sum(gt_dem, dtype=np.float64)
        sum_sq_val += np.sum(gt_dem.astype(np.float64) ** 2)
        count += gt_dem.size

    global_mean = sum_val / count
    global_std = np.sqrt((sum_sq_val / count) - (global_mean ** 2))

    return {
        "global_min": global_min,
        "global_max": global_max,
        "global_mean": global_mean,
        "global_std": global_std,
        "dem_range": (dem_min, dem_max),
        "gt_range": (gt_min, gt_max),
        "lidar_range": (lidar_min, lidar_max)
    }

stats = calculate_dataset_stats(PARENT_DIR)

print("="*50)
print("           ELEVATION DATASET STATISTICS           ")
print("="*50)
print(f"Global Min Elevation : {stats['global_min']:.4f} meters")
print(f"Global Max Elevation : {stats['global_max']:.4f} meters")
print(f"Global Mean          : {stats['global_mean']:.4f} meters")
print(f"Global Std Dev       : {stats['global_std']:.4f} meters")
print("-" * 50)
print(f"Bicubic DEM Range    : [{stats['dem_range'][0]:.2f}, {stats['dem_range'][1]:.2f}]")
print(f"Ground Truth Range   : [{stats['gt_range'][0]:.2f}, {stats['gt_range'][1]:.2f}]")
if stats['lidar_range'][0] != np.inf:
    print(f"Valid LiDAR Range    : [{stats['lidar_range'][0]:.2f}, {stats['lidar_range'][1]:.2f}]")
print("="*50)

Found 1655 files in 'D:\Vaibhav\vdsr-atl08\Dataset' and its subdirectories. Calculating statistics...


Scanning Dataset:   0%|          | 0/1655 [00:00<?, ?it/s]

           ELEVATION DATASET STATISTICS           
Global Min Elevation : 68.8346 meters
Global Max Elevation : 4312.1895 meters
Global Mean          : 1321.6211 meters
Global Std Dev       : 453.2252 meters
--------------------------------------------------
Bicubic DEM Range    : [68.83, 4306.93]
Ground Truth Range   : [69.65, 4312.19]
Valid LiDAR Range    : [114.79, 4230.83]


In [4]:
import os
import numpy as np
from tqdm.notebook import tqdm

# UPDATE THIS PATH to your parent directory
PARENT_DIR = r"D:\Vaibhav\vdsr-atl08\Dataset"

def analyze_photon_distribution(parent_dir):
    filepaths = []
    for root, _, files in os.walk(parent_dir):
        for f in files:
            if f.endswith('.npy'):
                filepaths.append(os.path.join(root, f))
                
    print(f"Found {len(filepaths)} files in '{parent_dir}'. Analyzing photon counts...")
    
    # Initialize dictionary to hold counts for each bin
    bins = {
        "0-10": 0,
        "11-30": 0,
        "31-50": 0,
        "51-100": 0,
        "101-250": 0,  # Adjusted to 101 to prevent overlap with 51-100
        "250+": 0
    }
    
    for path in tqdm(filepaths, desc="Scanning Photon Masks"):
        # Load data; shape is expected to be (4, H, W)
        data = np.load(path).astype(np.float32)
        
        # Channel 2 is the photon existence mask
        mask = data[2]
        
        # Count the number of valid photons (where mask is greater than 0)
        photon_count = np.sum(mask > 0)
        
        # Sort into appropriate bin
        if photon_count <= 10:
            bins["0-10"] += 1
        elif photon_count <= 30:
            bins["11-30"] += 1
        elif photon_count <= 50:
            bins["31-50"] += 1
        elif photon_count <= 100:
            bins["51-100"] += 1
        elif photon_count <= 250:
            bins["101-250"] += 1
        else:
            bins["250+"] += 1

    return bins, len(filepaths)

# Run the calculation
photon_bins, total_files = analyze_photon_distribution(PARENT_DIR)

# Print the neatly formatted results
print("="*50)
print("           PHOTON COUNT DISTRIBUTION              ")
print("="*50)
if total_files > 0:
    for bin_range, count in photon_bins.items():
        percentage = (count / total_files) * 100
        print(f"Range [{bin_range:>7} photons] : {count:<6} files  ({percentage:>5.1f}%)")
else:
    print("No files found to analyze.")
print("-" * 50)
print(f"Total Files Scanned      : {total_files}")
print("="*50)

Found 1655 files in 'D:\Vaibhav\vdsr-atl08\Dataset'. Analyzing photon counts...


Scanning Photon Masks:   0%|          | 0/1655 [00:00<?, ?it/s]

           PHOTON COUNT DISTRIBUTION              
Range [   0-10 photons] : 480    files  ( 29.0%)
Range [  11-30 photons] : 300    files  ( 18.1%)
Range [  31-50 photons] : 177    files  ( 10.7%)
Range [ 51-100 photons] : 264    files  ( 16.0%)
Range [101-250 photons] : 256    files  ( 15.5%)
Range [   250+ photons] : 178    files  ( 10.8%)
--------------------------------------------------
Total Files Scanned      : 1655


In [5]:
from dataloader_anchor import create_dataloaders
from model_symgatevdsr import SymGateVDSR, SymGateTopoLoss

train_loader, val_loader = create_dataloaders(TRAIN_DIRS, VAL_DIRS, batch_size=4, num_workers=0)
batch = next(iter(train_loader))

model = SymGateVDSR()
criterion = SymGateTopoLoss()

dem_bic, lidar_delta, mask, dist_map, gt_dem = (
    batch["dem_bic"], batch["lidar_delta"], batch["mask"], batch["dist_map"], batch["gt_dem"]
)
lidar_raw = dem_bic + lidar_delta

outputs = model(dem_bic, lidar_delta, mask, dist_map)
loss_dict = criterion(outputs, gt_dem, lidar_raw, lidar_delta, mask, dist_map)
print({k: v.item() for k, v in loss_dict.items()})
print("dem_pred == lidar_raw at masked pixels (should be ~0):",
      (mask * (outputs["dem_pred"] - lidar_raw).abs()).sum().item())

NameError: name 'VAL_DIRS' is not defined